# TP — Atterrissage sur un astéroïde : Lyapunov, ISS et backstepping

**Commande non linéaire — Master 3EA, parcours ASI — Université Claude Bernard Lyon 1**

Ce notebook accompagne l'énoncé `TP_Asteroide.pdf`. Il **est** votre rapport : répondez aux questions dans les cellules de texte, commentez chaque figure.

> **Sur Colab :** commencez par *Fichier > Enregistrer une copie dans Drive*.

Les cellules marquées `# À COMPLÉTER` sont à écrire.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.spatial.transform import Rotation

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (12, 3.4)

## 0. Le modèle (fourni)

Repère du site (origine au site, $z$ vers le haut) et repère de la sonde (caméra le long de $e_1$). La matrice $R$ envoie les coordonnées du repère du site dans celles du repère de la sonde. État : $X = -R\,P$ (position du site vue de la sonde), $V = -R\,\dot P$, $R$ et $\Omega$ (vitesse de rotation de la sonde).

$$\dot X = V + X\times\Omega,\qquad \dot V = \gamma - u_T + V\times\Omega,\qquad \dot R = -[\Omega]_\times R,\qquad \dot\Omega = u_C + j(\Omega),$$

avec $\gamma = g_0\,R\,e_3$ (gravité vue de la sonde) et $j(\Omega) = -J^{-1}(\Omega\times J\Omega)$.

**En Python :** `np.cross(a, b)` calcule le produit vectoriel $a\times b$, `a @ b` le produit scalaire de deux vecteurs (ou un produit matrice-vecteur), `np.linalg.norm(a)` la norme $|a|$, et `R.T` la transposée de `R`.

In [ ]:
g0 = 1.5e-4                                   # gravité au site (m/s²), connue à +-50 %
J = np.diag([400.0, 500.0, 600.0])            # inertie (kg.m²)
Ji = np.linalg.inv(J)
uT_max, uC_max = 0.02, 0.01                   # limites par axe : m/s² et rad/s²
e1, e3 = np.array([1.0, 0, 0]), np.array([0, 0, 1.0])

def hat(w):
    return np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])

def j(W):
    return -Ji @ np.cross(W, J @ W)

def decompose(x):
    return x[0:3], x[3:6], x[6:15].reshape(3, 3), x[15:18]

def etat_initial(P0=(20.0, 0.0, 100.0), Pdot0=(0.05, -0.02, -0.1), ecart_camera_deg=30.0,
                 Omega0=(0.01, -0.02, 0.01)):
    """Sonde en P0 (m, repère du site), caméra écartée du site de ecart_camera_deg."""
    P0, Pdot0 = np.asarray(P0, float), np.asarray(Pdot0, float)
    s_site = -P0 / np.linalg.norm(P0)
    r0 = Rotation.align_vectors([e1], [s_site])[0]                     # caméra sur le site
    R = (Rotation.from_euler('z', ecart_camera_deg, degrees=True) * r0).as_matrix()
    return np.r_[R @ (-P0), R @ (-Pdot0), R.ravel(), np.asarray(Omega0, float)]

def dynamique(t, x, loi, g_vrai):
    X, V, R, W = decompose(x)
    uT, uC = loi(x)
    uT, uC = np.clip(uT, -uT_max, uT_max), np.clip(uC, -uC_max, uC_max)
    return np.r_[V + np.cross(X, W), g_vrai * R @ e3 - uT + np.cross(V, W),
                 (-hat(W) @ R).ravel(), uC + j(W)]

def altitude(t, x, loi, g_vrai):
    X, V, R, W = decompose(x)
    return (-R.T @ X)[2]
altitude.terminal, altitude.direction = True, -1

def simule(loi, x0, g_vrai=g0, T=3000.0):
    sol = solve_ivp(dynamique, (0, T), x0, args=(loi, g_vrai), events=altitude,
                    max_step=1.0, rtol=1e-8, atol=1e-10)
    return sol

def bilan(sol, loi, nom=''):
    """Contact, vitesse et écart au contact, angle caméra/site, commandes."""
    X, V, R, W = decompose(sol.y[:, -1])
    P, Pdot = -R.T @ X, -R.T @ V
    touche = len(sol.t_events[0]) > 0
    d = np.linalg.norm(sol.y[0:3], axis=0)
    angle = np.degrees(np.arccos(np.clip(sol.y[0] / d, -1, 1)))
    U = np.array([np.r_[loi(sol.y[:, k])] for k in range(sol.y.shape[1])])
    print('%s contact : %s à t = %.0f s | vitesse %.3f m/s | écart latéral %.2f m | altitude finale %.3f m'
          % (nom, 'oui' if touche else 'non', sol.t[-1], np.linalg.norm(Pdot), np.linalg.norm(P[:2]), P[2]))
    return d, angle, U

def trace(sol, loi, titre=''):
    d, angle, U = bilan(sol, loi, titre)
    P = np.array([-decompose(sol.y[:, k])[2].T @ sol.y[0:3, k] for k in range(sol.y.shape[1])])
    fig, ax = plt.subplots(1, 4)
    ax[0].plot(P[:, 0], P[:, 2]); ax[0].set(xlabel='x (m)', ylabel='altitude (m)', title=titre)
    ax[1].plot(sol.t, d); ax[1].set(xlabel='t (s)', ylabel='distance au site (m)')
    ax[2].plot(sol.t, angle); ax[2].axhline(10, color='r', ls=':'); ax[2].set(xlabel='t (s)', ylabel='angle caméra / site (deg)')
    ax[3].plot(sol.t, np.abs(U[:, :3]).max(1), label='max |u_T|')
    ax[3].plot(sol.t, np.abs(U[:, 3:]).max(1), label='max |u_C|'); ax[3].legend(fontsize=8); ax[3].set(xlabel='t (s)')
    for a in ax:
        a.grid(True)
    plt.tight_layout(); plt.show()

## Animation (fournie)
La fonction `anime` transforme une simulation en petit film. Elle sert à *voir* ce que fait la loi de commande ; elle n'est pas à modifier, mais vous pouvez l'appeler sur n'importe laquelle de vos simulations (utile pour la soutenance).

In [ ]:
from matplotlib import animation
from matplotlib import patches as mpatches
from IPython.display import HTML

def _film(fig, dessine, n_images, ips=20):
    """Fabrique le film : vidéo si ffmpeg est disponible (c'est le cas sur Colab), lecteur intégré sinon."""
    anim = animation.FuncAnimation(fig, dessine, frames=n_images, interval=1000 / ips, blit=False)
    plt.close(fig)
    try:
        return HTML(anim.to_html5_video())
    except Exception:
        return HTML(anim.to_jshtml())

def _instants(t, n_images):
    """Indices de n_images instants régulièrement espacés (le solveur, lui, a un pas variable)."""
    return np.clip(np.searchsorted(t, np.linspace(t[0], t[-1], n_images)), 0, len(t) - 1)

def anime(sol, sol2=None, titre='', n_images=120):
    """Film de la mission, vue de côté dans le plan (x, z) du site (fourni).
    sol : sortie de simule ; sol2 : phase de pose éventuelle. À droite : zoom sur les 14 derniers mètres.
    Le cône est le champ de la caméra : vert si le site est dedans, rouge sinon."""
    t, Y = sol.t, sol.y
    n1 = len(t)
    if sol2 is not None:
        t, Y = np.r_[t, sol2.t], np.c_[Y, sol2.y]
    k = _instants(t, n_images)
    R = Y[6:15].T.reshape(-1, 3, 3)
    P = -np.einsum('nji,jn->ni', R, Y[0:3])                       # position de la sonde, repère du site
    cam = R[:, 0, :]                                              # direction de la caméra, repère du site
    d = np.linalg.norm(Y[0:3], axis=0)
    angle = np.degrees(np.arccos(np.clip(Y[0] / d, -1, 1)))
    vit = np.linalg.norm(Y[3:6], axis=0)
    fig, axs = plt.subplots(1, 2, figsize=(9.5, 5.2), gridspec_kw={'width_ratios': [1.25, 1]})
    rng = np.random.default_rng(1)
    for ax, (xl, zl), taille in zip(axs, (((-68, 68), (-8, 112)), ((-7.5, 7.5), (-1.5, 14.5))), (5.0, 0.9)):
        ax.set_facecolor('#0b1026'); ax.set(xlim=xl, ylim=zl, xlabel='x (m)'); ax.set_aspect('equal')
        ax.scatter(rng.uniform(*xl, 60), rng.uniform(0.15 * zl[1], zl[1], 60), s=rng.uniform(0.2, 3, 60), color='w')
        xs = np.linspace(*xl, 200)
        ax.fill_between(xs, zl[0], 0.012 * (zl[1] - zl[0]) * (np.sin(1.7 * xs / taille) + np.sin(0.6 * xs / taille + 1)) * (np.abs(xs) > taille), color='#6b5b4b')
        ax.plot(0, 0, 'x', color='r', ms=9, mew=2, zorder=6)
    axs[0].set(ylabel='altitude (m)', title=titre); axs[1].set(title='zoom près du site')
    objets = []
    for ax, taille in zip(axs, (5.0, 0.9)):
        sillage, = ax.plot([], [], color='c', lw=0.8, alpha=0.7)
        cone = mpatches.Polygon(np.zeros((3, 2)), closed=True, alpha=0.3, zorder=3); ax.add_patch(cone)
        corps = mpatches.Polygon(np.zeros((4, 2)), closed=True, fc='gold', ec='k', zorder=5); ax.add_patch(corps)
        pan = [mpatches.Polygon(np.zeros((4, 2)), closed=True, fc='royalblue', ec='w', lw=0.5, zorder=4) for _ in range(2)]
        for p in pan:
            ax.add_patch(p)
        objets.append((ax, taille, sillage, cone, corps, pan))
    texte = axs[0].text(0.03, 0.97, '', transform=axs[0].transAxes, va='top', family='monospace', fontsize=9, color='w')
    bulle = axs[1].text(0.5, 0.93, '', transform=axs[1].transAxes, ha='center', fontsize=12, weight='bold', color='w')
    carre = np.array([[-0.5, -0.5], [0.5, -0.5], [0.5, 0.5], [-0.5, 0.5]])
    contact = P[-1, 2] < 0.02

    def dessine(n):
        i = k[n]
        px, pz = P[i, 0], P[i, 2]
        c = np.array([cam[i, 0], cam[i, 2]])                      # caméra projetée dans le plan de la figure
        a = np.arctan2(c[1], c[0])
        Rz = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
        for ax, taille, sillage, cone, corps, pan in objets:
            pose = lambda pts: (taille * pts) @ Rz.T + [px, pz]
            corps.set_xy(pose(carre))
            for p, s in zip(pan, (-1, 1)):
                p.set_xy(pose(carre * [0.35, 0.9] + [0, s * 1.05]))
            L = min(d[i], 30.0 if taille > 1 else 9.0) * max(np.linalg.norm(c), 0.15)
            u, w = c / max(np.linalg.norm(c), 1e-9), np.tan(np.deg2rad(10.0))
            perp = np.array([-u[1], u[0]])
            cone.set_xy(np.array([[px, pz], [px, pz] + L * (u + w * perp), [px, pz] + L * (u - w * perp)]))
            cone.set_facecolor('lime' if angle[i] < 10 else 'red')
            sillage.set_data(P[:i + 1, 0], P[:i + 1, 2])
        phase = '' if sol2 is None else ('rendez-vous' if i < n1 else 'pose')
        texte.set_text('t = %5.0f s  %s\naltitude  %6.2f m\nvitesse   %6.3f m/s\ncaméra/site %4.0f°' % (t[i], phase, pz, vit[i], angle[i]))
        if contact and n == n_images - 1:
            v = 100 * vit[-1]
            bulle.set_text('contact à %.1f cm/s%s' % (v, '' if v <= 5 else ' : rebond !')); bulle.set_color('lime' if v <= 5 else 'red')
        elif angle[i] < 10:
            bulle.set_text('site dans le champ'); bulle.set_color('lime')
        else:
            bulle.set_text('site hors champ'); bulle.set_color('salmon')
    return _film(fig, dessine, n_images, ips=15)

# Séance 1 — Translation : rendez-vous, puis pose

## Travail préparatoire (questions 1 à 3 de l'énoncé)

Ces questions se traitent sur papier, **avant** de programmer. Rédigez ensuite vos réponses ici : elles font partie du rapport. Les formules s'écrivent en LaTeX entre `$...$` (double-cliquez sur une cellule de l'énoncé pour voir des exemples). Donnez les étapes du calcul, pas seulement le résultat.

### Q1. Équations dans le repère sonde

*Votre réponse :*

### Q2. Une fonction de Lyapunov indépendante de l'attitude

*Votre réponse :*

### Q3. Gravité mal connue

*Votre réponse :*

## Travail pratique

### Q4. Chute libre

In [ ]:
loi_nulle = lambda x: (np.zeros(3), np.zeros(3))
sol = simule(loi_nulle, etat_initial())
trace(sol, loi_nulle, 'chute libre')

In [ ]:
anime(sol, titre='chute libre')

*Votre commentaire :*

### Q5. Rendez-vous au point stationnaire
$u_T = \hat\gamma + k_1 (X + R P_c) + k_2 V$, avec $\hat\gamma = \hat g_0 R e_3$ et $P_c$ le point visé (repère du site). Rendez-vous : $P_c = (0, 0, 10)$ m. L'attitude est seulement amortie ici.

In [ ]:
P_rdv = np.array([0.0, 0.0, 10.0])

def loi_translation(x, P_c, k1=1e-4, k2=0.02, g_hat=g0):
    X, V, R, W = decompose(x)
    # À COMPLÉTER
    return np.zeros(3)

def amortit(x, k4=0.3):
    W = decompose(x)[3]
    return -j(W) - k4 * W

for r in (0.5, 1.0, 1.5):
    loi = lambda x, r=r: (loi_translation(x, P_rdv, g_hat=r * g0), amortit(x))
    sol = simule(loi, etat_initial(), T=3000)
    X, V, R, W = decompose(sol.y[:, -1])
    print('g_hat = %.1f g0 : position finale %s m, vitesse %.1e m/s' % (r, np.round(-R.T @ X, 2), np.linalg.norm(V)))
loi = lambda x: (loi_translation(x, P_rdv), amortit(x))
trace(simule(loi, etat_initial(), T=3000), loi, 'rendez-vous à 10 m')

**Vérification de Lyapunov.** Calculez $W_1=\tfrac{k_1}2|X+RP_c|^2+\tfrac12|V|^2$ le long du rendez-vous, pour $\hat g_0=g_0$ puis $\hat g_0=0{,}5\,g_0$.

In [ ]:
def W1_trajectoire(sol, P_c, k1=1e-4):
    """W1 = k1/2 |X + R P_c|^2 + 1/2 |V|^2 le long d'une simulation (tableau de même taille que sol.t)."""
    # À COMPLÉTER (decompose(sol.y[:, n]) donne X, V, R, Omega à l'instant n)
    return np.ones(sol.t.size)

plt.figure()
for r in (1.0, 0.5):
    loi = lambda x, r=r: (loi_translation(x, P_rdv, g_hat=r * g0), amortit(x))
    sol = simule(loi, etat_initial(), T=3000)
    W1 = W1_trajectoire(sol, P_rdv)
    plt.semilogy(sol.t, W1, label='g_hat = %.1f g0' % r)
    print('g_hat = %.1f g0 : W1 final = %.1e ; plus grande hausse relative entre deux instants : %.1e' % (r, W1[-1], np.max(np.diff(W1) / W1[:-1])))
plt.xlabel('t (s)'); plt.ylabel('W1'); plt.grid(True); plt.legend(); plt.title('Fonction de Lyapunov de la translation'); plt.show()


*Votre commentaire (lien avec la question 3 ; $W_1$ décroît-elle dans les deux cas ? vers quelle valeur ?) :*

### Q6. Bascule et loi de pose
Pendant le rendez-vous, on bascule vers la loi de pose dès que la condition `condition_bascule(x)` devient négative. La loi de pose vise un point $P_c = (0, 0, -h_a)$, à $h_a$ sous le site, avec ses propres gains.

In [ ]:
def simule_deux_phases(loi_rdv, loi_pose, condition_bascule, x0, g_vrai=g0, T=6000.0):
    """Rendez-vous, puis pose dès que condition_bascule(x) < 0 (fourni)."""
    def bascule(t, x, loi, g):
        return condition_bascule(x)
    bascule.terminal, bascule.direction = True, -1
    sol1 = solve_ivp(dynamique, (0, T), x0, args=(loi_rdv, g_vrai), events=[bascule, altitude],
                     max_step=1.0, rtol=1e-8, atol=1e-10)
    if len(sol1.t_events[0]) == 0:
        return sol1, None
    sol2 = solve_ivp(dynamique, (sol1.t[-1], sol1.t[-1] + T), sol1.y[:, -1], args=(loi_pose, g_vrai),
                     events=altitude, max_step=1.0, rtol=1e-8, atol=1e-10)
    return sol1, sol2

def condition_bascule(x, tol_P=1.5, tol_V=0.01):
    X, V, R, W = decompose(x)
    # À COMPLÉTER : négatif quand la sonde est à moins de tol_P du point de rendez-vous
    # et va moins vite que tol_V
    return 1.0

In [ ]:
print(' h_a   k1 pose  g_hat/g0  bascule (s)  contact (s)  vitesse (m/s)  écart (m)')
for h_a, k1p, k2p in ((2.0, 1e-4, 0.02), (1.0, 1e-4, 0.02), (1.0, 4e-4, 0.04), (0.5, 4e-4, 0.04)):
    for r in (0.5, 1.0, 1.5):
        loi_rdv = lambda x, r=r: (loi_translation(x, P_rdv, g_hat=r * g0), amortit(x))
        loi_pose = lambda x, r=r, h_a=h_a, k1p=k1p, k2p=k2p: (
            loi_translation(x, np.array([0, 0, -h_a]), k1=k1p, k2=k2p, g_hat=r * g0), amortit(x))
        sol1, sol2 = simule_deux_phases(loi_rdv, loi_pose, condition_bascule, etat_initial())
        if sol2 is None:
            print('%4.1f   %6.0e     %.1f      pas de bascule' % (h_a, k1p, r)); continue
        X, V, R, W = decompose(sol2.y[:, -1])
        P, Pdot = -R.T @ X, -R.T @ V
        print('%4.1f   %6.0e     %.1f       %5.0f        %5.0f       %6.3f       %5.2f' % (
            h_a, k1p, r, sol1.t[-1], sol2.t[-1], np.linalg.norm(Pdot), np.linalg.norm(P[:2])))

*Votre choix (h_a, gains de pose, seuils de bascule), justifié par le cahier des charges :*

# Séance 2 — Pointage de la caméra par backstepping

## Travail préparatoire (questions 7 à 9 de l'énoncé)

Ces questions se traitent sur papier, **avant** de programmer. Rédigez ensuite vos réponses ici : elles font partie du rapport. Les formules s'écrivent en LaTeX entre `$...$` (double-cliquez sur une cellule de l'énoncé pour voir des exemples). Donnez les étapes du calcul, pas seulement le résultat.

### Q7. Cinématique de la visée

*Votre réponse :*

### Q8. Backstepping en deux étapes

*Votre réponse :*

### Q9. Ce qui ne peut pas être global

*Votre réponse :*

## Travail pratique

### Q10. La loi d'attitude
$s = X/|X|$, $\omega_{los} = s\times V/|X|$, $\dot s = (\omega_{los} - \Omega)\times s$.

* étape 1 : $\Omega_d = \omega_{los} - k_3\,(s\times e_1)$ ;
* étape 2 : $e_\Omega = \Omega - \Omega_d$ et $u_C = -j(\Omega) + \dot\Omega_d - \mu\,(s\times e_1) - k_4 e_\Omega$.

Pour $\dot\Omega_d$ : $\dot\omega_{los} = (\dot s\times V + s\times\dot V)/|X| - \omega_{los}\,(s\cdot V)/|X|$, avec $\dot V = \hat\gamma - u_T + V\times\Omega$ (tout est connu).
Le pointage sert pendant le rendez-vous ; pendant la pose, l'attitude est simplement amortie.

In [ ]:
def loi_complete(x, k3=0.05, k4=0.3, mu=0.01, g_hat=g0, P_c=P_rdv, k1=1e-4, k2=0.02, d_coupure=0.0):
    """Translation vers P_c et pointage de la caméra sur le site (coupé sous d_coupure)."""
    X, V, R, W = decompose(x)
    uT = loi_translation(x, P_c, k1=k1, k2=k2, g_hat=g_hat)
    nX = np.linalg.norm(X)
    if nX <= d_coupure:
        return uT, amortit(x, k4)
    s = X / nX
    uT_appl = np.clip(uT, -uT_max, uT_max)              # poussée réellement appliquée
    # À COMPLÉTER : w_los, Omega_d, s_point, V_point, w_los_point, Omega_d_point, puis uC
    uC = -j(W) - k4 * W
    return uT, uC

In [ ]:
for mu in (1.0, 0.01):
    loi = lambda x, mu=mu: loi_complete(x, mu=mu)
    sol = simule(loi, etat_initial(), T=1000)
    d, angle, U = bilan(sol, loi, 'rendez-vous avec pointage, mu = %g :' % mu)
    hors = np.where(angle > 10)[0]
    print('    site dans le champ à partir de t = %.0f s ; u_C saturée pendant %.0f s'
          % (sol.t[hors[-1]] if len(hors) else 0, np.sum(np.diff(sol.t)[np.abs(U[:-1, 3:]).max(1) >= uC_max - 1e-12])))
    trace(sol, loi, 'mu = %g' % mu)

*Votre commentaire : rôle de $\mu$, temps d'acquisition (angle < 10°), saturation de $u_C$.*

### Q11. Départ caméra presque à l'opposé du site

In [ ]:
for ecart in (90, 150, 179):
    sol = simule(loi_complete, etat_initial(ecart_camera_deg=ecart, Omega0=(0, 0, 0)), T=1000)
    d = np.linalg.norm(sol.y[0:3], axis=0)
    angle = np.degrees(np.arccos(np.clip(sol.y[0] / d, -1, 1)))
    hors = np.where(angle > 10)[0]
    print('écart initial %3d° : site dans le champ à partir de t = %.0f s' % (ecart, sol.t[hors[-1]] if len(hors) else 0))

*Votre commentaire :*

### Q12. Mission complète, et pointage pendant la pose ?
Rendez-vous avec pointage, bascule, puis pose avec $h_a = 1$ m et des gains $(4\cdot10^{-4};\ 0{,}04)$. On compare une pose à attitude amortie et une pose où l'on garde le pointage jusqu'à 1 m, puis 0,2 m du site.

In [ ]:
P_pose = np.array([0.0, 0.0, -1.0])
loi_rdv = lambda x: loi_complete(x)
for d_c in (None, 1.0, 0.2):
    if d_c is None:
        loi_pose = lambda x: (loi_translation(x, P_pose, k1=4e-4, k2=0.04), amortit(x))
    else:
        loi_pose = lambda x, d_c=d_c: loi_complete(x, P_c=P_pose, k1=4e-4, k2=0.04, d_coupure=d_c)
    sol1, sol2 = simule_deux_phases(loi_rdv, loi_pose, condition_bascule, etat_initial())
    X, V, R, W = decompose(sol2.y[:, -1])
    d = np.linalg.norm(sol2.y[0:3], axis=0)
    print('pose %-28s : contact à %4.0f s, %.3f m/s, écart %.2f m ; max |Omega| sous 5 m = %.3f rad/s'
          % ('attitude amortie' if d_c is None else 'pointage jusqu\'à %.1f m' % d_c, sol2.t[-1],
             np.linalg.norm(V), np.linalg.norm((-R.T @ X)[:2]), np.abs(sol2.y[15:18, d < 5]).max()))

In [ ]:
# Film de la dernière mission simulée (rendez-vous avec pointage, puis pose)
anime(sol1, sol2, titre='mission complète')

*Votre commentaire :*

### Question ouverte (une au choix, obligatoire)
Choisissez une des trois questions de l'énoncé (caméra pixelisée et champ limité, rendez-vous précis, vitesse estimée) et traitez-la ici. Elle n'est pas guidée et doit apparaître dans la soutenance.

In [ ]:
# Votre travail